In [15]:
import pandas as pd
import numpy as np
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts
from pyecharts.charts import *
from pyecharts.commons.utils import JsCode
from pyecharts.globals import ThemeType
from pyecharts import options as opts
from pyecharts.charts import Bar
from pyecharts.faker import Faker
#导入必要的工具包

In [2]:
data = pd.read_csv('./淘宝用户行为.csv')
#读取数据

In [3]:
data.info() 
#查看数据基本信息

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12256906 entries, 0 to 12256905
Data columns (total 6 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   user_id        int64 
 1   item_id        int64 
 2   behavior_type  int64 
 3   user_geohash   object
 4   item_category  int64 
 5   time           object
dtypes: int64(4), object(2)
memory usage: 561.1+ MB


In [4]:
data.isnull().sum() 
#查看为空值的数据，地理数据缺失

user_id                0
item_id                0
behavior_type          0
user_geohash     8334824
item_category          0
time                   0
dtype: int64

In [5]:
del data['user_geohash'] 
#删除地理位置数据不进行分析

In [6]:
#日期转化拆成 日期和小时
data['date'] = data.time.apply(lambda x:x.split(' ')[0])
#将time，date转为datetime格式
data['time'] = pd.to_datetime(data['time'])
data['date'] = pd.to_datetime(data['date'])

## 用户路径行为分析

In [7]:
SC = data[data['behavior_type']==2]
JG = data[data['behavior_type']==3]

In [8]:
SC = SC.reset_index(drop=True) #重置索引
JG = JG.reset_index(drop=True)

In [9]:
SC_G = SC.groupby('date')['user_id'].count() #按照日期做数据透视表，一天每个ID重复收藏加购也算
JG_G = JG.groupby('date')['user_id'].count()

In [10]:
line = Line()
line.add_xaxis(SC_G.index.astype('str').to_list())
line.add_yaxis("收藏",SC_G.values.tolist(),is_smooth=False,is_symbol_show=True)
line.add_yaxis("加购",JG_G.values.tolist(),is_smooth=False,is_symbol_show=True,markline_opts=opts.MarkLineOpts(data=[opts.MarkLineItem(type_="average")]))
line.set_global_opts(title_opts=opts.TitleOpts(title="收藏加购量折线图", subtitle="双十二前后收藏量和加购量"))
line.render_notebook()

**双十二之前,收藏和加购呈现缓慢上升的趋势,双十二当天是数值顶峰,然后就开始回归平常 这个店铺加购比收藏多25%,属于正常现象,收藏和加购随着双12逐渐增加,购物车在双1212当天增长最明显,反观收藏就增长较弱**

## 支付行为的走势变化

In [11]:
PAY = data[data['behavior_type']==4].copy()
PAY.reset_index(drop=True,inplace=True)

In [12]:
PAY['date']=PAY.date.astype('str')  #转置日期格式为字符串
PAY['date']=PAY['date'].str.split('2014-').str[1] #日期缩短

In [13]:
PAY_G = PAY.groupby('date').agg({'user_id':'count'}).reset_index()

In [16]:
bar2 = Bar()
bar2.add_xaxis(PAY_G.date.tolist())
bar2.add_yaxis("",PAY_G.user_id.tolist(), color=Faker.rand_color())
bar2.set_global_opts(title_opts=opts.TitleOpts(title="支付行为走势图", subtitle="双十二前后支付行为"))
bar2.set_global_opts(
        title_opts=opts.TitleOpts(title="支付人数柱状图", subtitle="双十二支付行为滚动柱状图,鼠标操作可调节时间区间"),
        datazoom_opts=opts.DataZoomOpts(type_="inside"),
    )
bar2.render_notebook()

   ### 店铺日常的销售量差距不大,活动期当天爆发4倍,相对2020年行业来说处于较低的水平 可能这个店铺是主要做日常销售,日常和活动的价差并不是很大,如果本身利润较低或者供应链渠道问题,平稳递增是个好选择,如果本身利润率较高的话,是可以在大促期做一个高爆发高产出